# 03 · Demo Seminar 2: RAG giảm hallucination mà vẫn giữ accuracy

**Đề 11, RAG: Concepts & Applications.** Thông điệp: không phương pháp nào tốt nhất cho mọi trường hợp, phải chọn theo loại dữ liệu
và loại câu hỏi. Demo đi theo flow **LLM thuần, RAG và biến thể, KG-RAG** trên cùng một bài toán: hỏi đáp trên đồ thị tri thức
Freebase (WebQSP, 1000 câu test, cùng tập câu cho mọi cấu hình), hai reader Qwen2.5-7B-Instruct và Llama-3.1-8B-Instruct.

Notebook chỉ **đọc kết quả đã lưu** của notebook 01 và 02, không gọi LLM, không cần GPU. Nếu có `results/snapshot/` thì đọc từ đó,
nếu không thì đọc trên Drive.

Chỉ số dùng xuyên suốt:

| Chỉ số | Ý nghĩa |
|---|---|
| llm_hit | câu trả lời chứa ít nhất một đáp án đúng |
| grounded_correct | đúng **và** có đường đi trong đồ thị làm chứng; đây là chỉ số chính "đúng mà không bịa" |
| QHR | tỷ lệ câu trả lời có ít nhất một entity không có chứng cứ trong đồ thị (bịa) |
| strict_QHR | QHR tính thêm entity có trong đồ thị nhưng nối bằng quan hệ sai |
| AURC_incorrect | rủi ro trả lời sai khi xếp câu theo confidence của retriever; càng thấp càng biết lúc nào nên từ chối |
| #paths, tokens | số đường KG và số token đưa vào prompt (chi phí) |

In [1]:
# ==== Setup: chỉ đọc kết quả đã lưu, không cần GPU ====
# Nguồn kết quả, theo thứ tự ưu tiên:
#   1. results/snapshot/ trong folder này (tạo bằng cell kế tiếp hoặc python -m kgqa_bench.snapshot), chạy được không cần Drive
#   2. graph_rag/results_gnn/ trên Drive (kết quả đầy đủ của notebook 01, 02)
import os, sys, json, glob, re
import pandas as pd
try:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab'   # SỬA NẾU KHÁC
except ImportError:
    DRIVE_BASE = os.path.abspath('../..')                     # chạy local từ seminar2_kgrag/notebooks
CODE_DIR    = os.environ.get('S2_CODE_DIR', f'{DRIVE_BASE}/seminar2_kgrag')
SNAPSHOT    = f'{CODE_DIR}/results/snapshot'
DRIVE_RES   = f'{DRIVE_BASE}/graph_rag/results_gnn'
DRIVE_DATA  = f'gnnrag:{DRIVE_BASE}/graph_rag/ckpt/gnn_features_webqsp_test.pkl:webqsp:test'
RESULTS     = os.environ.get('S2_RESULTS') or (SNAPSHOT if os.path.exists(f'{SNAPSHOT}/MANIFEST.json') else DRIVE_RES)
DATA_SPEC   = os.environ.get('S2_DATA_SPEC', DRIVE_DATA)
OUT_DIR     = os.environ.get('S2_OUT_DIR', f'{CODE_DIR}/results')
sys.path.insert(0, CODE_DIR)
from kgqa_bench.data import load_spec
READERS = ['Qwen2.5-7B-Instruct', 'Meta-Llama-3.1-8B-Instruct']
SHORT = {'Qwen2.5-7B-Instruct': 'Qwen2.5-7B', 'Meta-Llama-3.1-8B-Instruct': 'Llama-3.1-8B', 'Mistral-7B-Instruct-v0.3': 'Mistral-7B'}
PREFIX = {'chat': 'b08', 'grounded': 'b08g', 'chat+verify': 'b08v', 'cand+verify': 'b08c', 'lora-cand+verify': 'b08f'}
pd.set_option('display.max_colwidth', 100); pd.set_option('display.width', 200)
print('results =', RESULTS)

Mounted at /content/drive
results = /content/drive/MyDrive/ThS/Research/colab/graph_rag/results_gnn


In [2]:
# ==== (Chỉ chạy một lần, trên Colab) tạo snapshot gọn từ Drive vào results/snapshot/ ====
# Sau đó tải folder seminar2_kgrag về máy: notebook này chạy được ở bất kỳ đâu, không cần Drive.
MAKE_SNAPSHOT = not os.path.exists(f'{SNAPSHOT}/MANIFEST.json') and os.path.exists(DRIVE_RES)
if MAKE_SNAPSHOT:
    from kgqa_bench.snapshot import make_snapshot
    make_snapshot(DRIVE_RES, DRIVE_DATA, SNAPSHOT)
    RESULTS = SNAPSHOT
print('snapshot:', json.load(open(f'{SNAPSHOT}/MANIFEST.json')) if os.path.exists(f'{SNAPSHOT}/MANIFEST.json') else 'chưa có')

snapshot -> /content/drive/MyDrive/ThS/Research/colab/seminar2_kgrag/results/snapshot: 87 files, 66.62 MB, 11 reader dirs, 996 questions
snapshot: {'created': '2026-11-08 18:27:51', 'source_results': '/content/drive/MyDrive/ThS/Research/colab/graph_rag/results_gnn', 'data_spec': 'gnnrag:/content/drive/MyDrive/ThS/Research/colab/graph_rag/ckpt/gnn_features_webqsp_test.pkl:webqsp:test', 'reader_dirs': ['b08_reader_Meta-Llama-3.1-8B-Instruct', 'b08_reader_Mistral-7B-Instruct-v0.3', 'b08_reader_Qwen2.5-7B-Instruct', 'b08c_reader_Meta-Llama-3.1-8B-Instruct', 'b08c_reader_Qwen2.5-7B-Instruct', 'b08f_reader_Meta-Llama-3.1-8B-Instruct', 'b08f_reader_Qwen2.5-7B-Instruct', 'b08g_reader_Meta-Llama-3.1-8B-Instruct', 'b08g_reader_Qwen2.5-7B-Instruct', 'b08v_reader_Meta-Llama-3.1-8B-Instruct', 'b08v_reader_Qwen2.5-7B-Instruct'], 'records_for_readers': ['Qwen2.5-7B-Instruct'], 'records_for_methods': ['no_retrieval', 'gnnrag_all', 'gnnrag_top5', 'sbert_top5', 'cgar', 'cgar_esc'], 'n_files': 87, 'n_que

In [3]:
# ==== Helper đọc kết quả ====
def _j(path):
    try:
        with open(path) as f: return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError): return None

def rdir(cfg, reader): return f'{RESULTS}/{PREFIX[cfg]}_reader_{reader}'

def summary(cfg, reader):
    R = _j(f'{rdir(cfg, reader)}/results.json')
    return next(iter(R['summaries'].values())) if R and R.get('summaries') else {}

def hsummary(cfg, reader):
    H = _j(f'{rdir(cfg, reader)}/hallucination/hallucination.json')
    return next(iter(H['datasets'].values()))['summaries'] if H and H.get('datasets') else {}

def _jsonl(path):
    if not path or not os.path.exists(path): return {}
    with open(path) as f: return {r['qid']: r for r in (json.loads(l) for l in f if l.strip())}

_CACHE = {}
def records(cfg, reader, method):
    k = ('r', cfg, reader, method)
    if k not in _CACHE:
        ps = sorted(glob.glob(f'{rdir(cfg, reader)}/records_*_{method}.jsonl')); _CACHE[k] = _jsonl(ps[0] if ps else None)
    return _CACHE[k]

def claims(cfg, reader, method):
    k = ('c', cfg, reader, method)
    if k not in _CACHE:
        ps = sorted(glob.glob(f'{rdir(cfg, reader)}/hallucination/claims_*_{method}.jsonl')); _CACHE[k] = _jsonl(ps[0] if ps else None)
    return _CACHE[k]

def row(label, cfg, reader, method):
    s, h = summary(cfg, reader).get(method, {}), hsummary(cfg, reader).get(method, {})
    if not s: return None
    return {'reader': SHORT.get(reader, reader), 'cấu hình': label, 'llm_hit': s.get('llm_hit'), 'llm_f1': s.get('llm_f1'),
            'grounded_correct': h.get('grounded_correct_rate'), 'QHR': h.get('question_hallucination_rate'),
            'strict_QHR': h.get('question_hallucination_rate_strict'), 'abstain': s.get('llm_abstained'),
            'AURC_incorrect': h.get('aurc_incorrect'), '#paths': s.get('cost'), 'tokens': s.get('n_tokens_prompt')}

def table(specs, cols=None):
    """specs = [(nhãn, cấu hình, reader, method)]; bỏ qua cấu hình chưa có trên Drive."""
    rows = [r for r in (row(*sp) for sp in specs) if r]
    if not rows:
        print('(chưa có kết quả cho các cấu hình này trong', RESULTS, ')'); return pd.DataFrame()
    df = pd.DataFrame(rows).set_index(['reader', 'cấu hình'])
    if cols: df = df[[c for c in cols if c in df.columns]]
    df = df.astype(float).dropna(axis=1, how='all')
    display(df.round(3)); return df

DS = None
def question(qid):
    global DS
    if DS is None:
        qpath = f'{RESULTS}/questions.json'
        if os.path.exists(qpath):        # snapshot: câu hỏi + gold đã trích sẵn
            from types import SimpleNamespace
            DS = {k: SimpleNamespace(qid=k, **v) for k, v in json.load(open(qpath)).items()}
        else:
            try: DS = {q.qid: q for q in load_spec(DATA_SPEC)}
            except Exception as e: print('không nạp được dataset (chỉ ảnh hưởng phần hiển thị câu hỏi/gold):', e); DS = {}
    return DS.get(qid)

def _short(t, n=240):
    t = ' '.join(str(t or '').split()); return t if len(t) <= n else t[:n] + ' ...'

def gold_path(qid, cfg='chat', reader=READERS[0], method='gnnrag_all'):
    """Một đường trong context của GNN-RAG-all kết thúc ở đáp án gold: chứng cứ trong đồ thị."""
    q, r = question(qid), records(cfg, reader, method).get(qid)
    if not q or not r: return None
    gold = {a.lower() for a in q.answers}
    return next((p for p in r.get('selected_paths', []) if p.split(' -> ')[-1].strip().lower() in gold), None)

def show_question(qid, specs):
    """In câu hỏi, gold, rồi câu trả lời của từng cấu hình kèm nhãn từng claim của verifier."""
    q = question(qid)
    print(f'[{qid}]', q.question if q else '')
    if q: print('   gold:', q.answers[:6])
    for label, cfg, rd, method in specs:
        r, v = records(cfg, rd, method).get(qid), claims(cfg, rd, method).get(qid)
        if not r: continue
        print(f'-- {label} | {SHORT.get(rd, rd)} | hit={r.get("llm_hit")} | đáp án có trong context={r.get("hit_ctx")}')
        raw = r.get('llm_pred_raw') or r.get('llm_pred')
        print('   trả lời:', _short(raw))
        if r.get('llm_pred_raw') and r.get('llm_pred') != r.get('llm_pred_raw'): print('   sau verify:', _short(r.get('llm_pred')))
        for c in (v or {}).get('claims', [])[:6]:
            print(f'     claim {_short(c["claim"], 45)!r:48s} {c["label"]:11s} {c["evidence_type"]:20s} gold={c["is_gold"]}')
        if v: print('   loại lỗi:', v.get('error_category'))

def pick(cond, k=1):
    """k qid đầu tiên (theo thứ tự qid, tái lập được) thoả cond(qid); duyệt trên tập câu của GNN-RAG-all chat."""
    base = records('chat', READERS[0], 'gnnrag_all')
    return [q for q in sorted(base) if cond(q)][:k]

## Phần 1 · LLM thuần

**1. Hoạt động thế nào.** LLM trả lời hoàn toàn từ tham số đã học khi pretrain hoặc fine-tune, không tra cứu nguồn ngoài.

**2. Ưu / nhược.** Ưu: nhanh, rẻ, không cần hạ tầng truy hồi; fine-tune học được các mẫu ổn định của một bài toán.
Nhược: tri thức đóng băng tại thời điểm train, không có chứng cứ để kiểm tra, và khi không biết thì vẫn trả lời trôi chảy (bịa).

**3. Hợp với dữ liệu và câu hỏi nào.** Bài toán có tập nhãn cố định mà tín hiệu nằm ngay trong đầu vào. Paper tham khảo phát hiện mã độc
trong package PyPI là ví dụ điển hình: đầu vào là chính mã nguồn package, đầu ra là "độc / không độc". Fine-tune dạy mô hình nhận ra mẫu
mã độc trực tiếp từ đầu vào, còn RAG chỉ kéo về các package "trông giống" nên thêm nhiễu và tốn context. Vì vậy paper thấy fine-tune hơn RAG.
Không hợp: câu hỏi mở về sự kiện cụ thể, thực thể đuôi dài, tri thức thay đổi. Đáp án ở đây là một thực thể cụ thể trong Freebase,
không suy ra được từ câu hỏi.

**4. Demo.** Bảng dưới cho thấy trên bài toán hỏi đáp tri thức, LLM thuần kém xa khi có retrieval, và fine-tune (LoRA) **không** cứu được
khi thiếu context: điều ngược với kết luận của paper PyPI, vì loại dữ liệu và loại câu hỏi khác nhau.

In [4]:
specs1 = []
for rd in READERS:
    specs1 += [('LLM thuần (zero-shot)', 'chat', rd, 'no_retrieval'),
               ('LLM thuần + fine-tune LoRA', 'lora-cand+verify', rd, 'no_retrieval'),
               ('GNN-RAG (có đường KG), prompt chat', 'chat', rd, 'gnnrag_all')]
T1 = table(specs1, cols=['llm_hit', 'llm_f1', 'tokens'])

llm_hit  llm_f1   tokens
reader       cấu hình                                                    
Qwen2.5-7B   LLM thuần (zero-shot)                 0.474   0.283   41.342
             LLM thuần + fine-tune LoRA            0.716   0.484  108.181
             GNN-RAG (có đường KG), prompt chat    0.765   0.625  239.068
Llama-3.1-8B LLM thuần (zero-shot)                 0.588   0.354   41.259
             LLM thuần + fine-tune LoRA            0.724   0.496  106.322
             GNN-RAG (có đường KG), prompt chat    0.805   0.630  232.797

LLM vẫn "nhớ" được một phần đáp án. Cột *đúng mà không có chứng cứ* đếm câu trả lời đúng nhưng không có đường đi nào trong đồ thị làm chứng:
đúng nhờ trí nhớ, không kiểm chứng được. Mistral có llm_hit cao nhưng gần 1/5 câu "đúng" thuộc loại này, nên llm_hit một mình
không đủ để đánh giá hallucination.

In [5]:
rows = []
for rd in READERS + ['Mistral-7B-Instruct-v0.3']:
    h = hsummary('chat', rd).get('gnnrag_all', {})
    if h:
        rows.append({'reader': SHORT.get(rd, rd), 'llm_hit': h.get('llm_hit'), 'grounded_correct': h.get('grounded_correct_rate'),
                     'đúng mà không có chứng cứ': h.get('correct_ungrounded_rate'), 'QHR': h.get('question_hallucination_rate')})
display(pd.DataFrame(rows).set_index('reader').astype(float).round(3)) if rows else print('(chưa có kết quả P9 cho prompt chat)')

,llm_hit,grounded_correct,đúng mà không có chứng cứ,QHR
reader,,,,
Qwen2.5-7B,0.765,0.722,0.043,0.114
Llama-3.1-8B,0.805,0.744,0.061,0.140
Mistral-7B,0.806,0.616,0.190,0.316


**Ví dụ LLM thuần bịa.** Câu mà LLM thuần trả lời sai (không phải từ chối), trong khi đồ thị có đường đi tới đáp án đúng.

In [6]:
rd = READERS[0]
nr, al = records('chat', rd, 'no_retrieval'), records('chat', rd, 'gnnrag_all')
EX1 = pick(lambda q: nr.get(q, {}).get('llm_hit') == 0 and not nr.get(q, {}).get('llm_abstained')
           and al.get(q, {}).get('llm_hit') == 1 and gold_path(q) is not None, k=3)
for qid in EX1:
    show_question(qid, [('LLM thuần', 'chat', rd, 'no_retrieval'), ('GNN-RAG, prompt chat', 'chat', rd, 'gnnrag_all')])
    print('   đường KG làm chứng:', gold_path(qid)); print()

[WebQTest-1020] what highschool did r. kelly attend
   gold: ['Kenwood Academy']
-- LLM thuần | Qwen2.5-7B | hit=0 | đáp án có trong context=0
   trả lời: -
-- GNN-RAG, prompt chat | Qwen2.5-7B | hit=1 | đáp án có trong context=1
   trả lời: ["Kenwood Academy"]
     claim 'Kenwood Academy'                                SUPPORTED   path_endpoint        gold=True
   loại lỗi: ok_grounded
   đường KG làm chứng: R. Kelly -> people.person.education -> m.0h3355s -> education.educational_institution.students_graduates -> Kenwood Academy

[WebQTest-1021] what happened at benghazi
   gold: ['2013 Benghazi conflict', 'First Battle of Benghazi', 'Operation Dignity Battle', '2012 Benghazi attack', '2012 U.S. diplomatic missions attacks', 'Battle of Benina Airport']
-- LLM thuần | Qwen2.5-7B | hit=0 | đáp án có trong context=0
   trả lời: - Attack on the U.S. consulate in Benghazi - Death of U.S. Ambassador to Libya Christopher Stevens
-- GNN-RAG, prompt chat | Qwen2.5-7B | hit=1 | đáp án có trong

## Phần 2 · RAG và biến thể

**1. Hoạt động thế nào.** Chuyển nguồn tri thức thành các đoạn văn bản, nhúng (embedding) câu hỏi và từng đoạn, lấy top-K đoạn giống câu
hỏi nhất đưa vào prompt. Ở đây mỗi "đoạn" là một đường trong đồ thị đã viết thành chữ, ví dụ
`Barack Obama -> people.person.spouse_s -> m.02h98sm -> ... -> Michelle Obama`, và retriever là SBERT top-5 (`sbert_top5`).
Biến thể so sánh: top-5 theo điểm GNN (`gnnrag_top5`); prompt *grounded* ép chỉ trả lời từ context và cho phép "unknown";
*verify* hậu kỳ chỉ giữ entity có trong context.

**2. Ưu / nhược.** Ưu: cập nhật tri thức không cần train lại, câu trả lời có nguồn. Nhược: giống bề mặt không đồng nghĩa với liên quan;
cắt top-K làm rơi đáp án ở câu nhiều bước hoặc nhiều đáp án; LLM vẫn chèn kiến thức ngoài context. Prompt grounded giảm bịa bằng cách
bỏ cuộc: Qwen từ chối gần 30% câu kể cả khi đáp án nằm sẵn trong context; Llama viết câu từ chối rồi vẫn thêm kiến thức riêng.

**3. Hợp với dữ liệu và câu hỏi nào.** Văn bản phi cấu trúc, câu hỏi một bước mà đáp án nằm gọn trong một đoạn, kho tài liệu thay đổi thường xuyên.

**4. Demo.** Bảng dưới, rồi một ví dụ RAG thất bại: top-5 theo độ tương đồng bỏ mất đường đúng, reader trả lời bằng thực thể không có
chứng cứ. Đó là chỗ cấu trúc đồ thị ở Phần 3 giúp được.

In [7]:
specs2 = []
for rd in READERS:
    specs2 += [('RAG dense top-5, prompt chat', 'chat', rd, 'sbert_top5'),
               ('GNN top-5, prompt chat', 'chat', rd, 'gnnrag_top5'),
               ('RAG dense top-5, prompt grounded', 'grounded', rd, 'sbert_top5'),
               ('RAG dense top-5, chat + verify', 'chat+verify', rd, 'sbert_top5')]
T2 = table(specs2, cols=['llm_hit', 'grounded_correct', 'QHR', 'strict_QHR', 'abstain', '#paths', 'tokens'])

llm_hit  grounded_correct    QHR  strict_QHR  abstain  #paths   tokens
reader       cấu hình                                                                                                
Qwen2.5-7B   RAG dense top-5, prompt chat        0.755             0.709  0.118       0.254    0.018   3.462  124.498
             GNN top-5, prompt chat              0.751             0.697  0.131       0.252    0.020   3.462  124.976
             RAG dense top-5, prompt grounded    0.631             0.620  0.028       0.163    0.286   3.462  146.498
             RAG dense top-5, chat + verify      0.756             0.756  0.001       0.209    0.018   3.462  124.498
Llama-3.1-8B RAG dense top-5, prompt chat        0.768             0.704  0.150       0.289    0.025   3.462  121.593
             GNN top-5, prompt chat              0.769             0.704  0.157       0.282    0.020   3.462  122.236
             RAG dense top-5, prompt grounded    0.721             0.594  0.194       0.322    0.150   3.462  143.593
             RAG dense top-5, chat + verify      0.761             0.760  0.001       0.245    0.025   3.462  121.593

In [8]:
rd = READERS[0]
sb, sbc, al = records('chat', rd, 'sbert_top5'), claims('chat', rd, 'sbert_top5'), records('chat', rd, 'gnnrag_all')
EX2 = pick(lambda q: sb.get(q, {}).get('hit_ctx') == 0 and sbc.get(q, {}).get('hallucinated') == 1
           and al.get(q, {}).get('llm_hit') == 1, k=2)
print('Ví dụ RAG top-5 thất bại: context thiếu đáp án, reader bịa\n')
for qid in EX2:
    show_question(qid, [('RAG dense top-5', 'chat', rd, 'sbert_top5'), ('GNN-RAG all', 'chat', rd, 'gnnrag_all')])
    print('   đường KG làm chứng:', gold_path(qid)); print()
gr = records('grounded', rd, 'sbert_top5')
EX2B = pick(lambda q: gr.get(q, {}).get('llm_abstained') == 1 and gr.get(q, {}).get('hit_ctx') == 1, k=1)
print('Ví dụ prompt grounded bỏ cuộc dù đáp án nằm trong context\n')
for qid in EX2B:
    show_question(qid, [('RAG dense top-5, grounded', 'grounded', rd, 'sbert_top5'), ('RAG dense top-5, chat', 'chat', rd, 'sbert_top5')])

Ví dụ RAG top-5 thất bại: context thiếu đáp án, reader bịa

[WebQTest-100] what language is spoken in haiti today
   gold: ['Haitian Creole', 'French']
-- RAG dense top-5 | Qwen2.5-7B | hit=1 | đáp án có trong context=0
   trả lời: - Haitian Creole - French
     claim 'Haitian Creole'                                 UNSUPPORTED in_context_no_proof  gold=True
     claim 'French'                                         UNSUPPORTED in_subgraph_only     gold=True
   loại lỗi: correct_ungrounded
-- GNN-RAG all | Qwen2.5-7B | hit=1 | đáp án có trong context=0
   trả lời: - Haitian Creole - French
     claim 'Haitian Creole'                                 UNSUPPORTED in_context_no_proof  gold=True
     claim 'French'                                         UNSUPPORTED in_subgraph_only     gold=True
   loại lỗi: correct_ungrounded
   đường KG làm chứng: None

[WebQTest-1091] what type of religions are in the united states
   gold: ['Hinduism', 'Unitarian Universalism', 'Buddhism', 'Christiani

## Phần 3 · KG-RAG

**1. Hoạt động thế nào.** Tri thức là đồ thị thực thể và quan hệ. GNN (ReaRev) suy luận trên subgraph quanh thực thể của câu hỏi và chấm
điểm các thực thể ứng viên; hệ thống trích các đường quan hệ từ thực thể chủ đề tới ứng viên và đưa cho LLM (**GNN-RAG**).
**CGAR** học chọn ít đường hơn (tối đa 5) và trả về một confidence cho cả câu. **CGAR-G** thêm ba bước nhắm thẳng vào hallucination:
(a) câu khó hoặc confidence thấp thì mở rộng sang toàn bộ đường; (b) prompt liệt kê các ứng viên là điểm cuối của đường;
(c) *verify*: chỉ giữ entity trùng một ứng viên, tức có đường đi làm chứng. Cuối cùng reader được fine-tune LoRA trên tập train.

**2. Ưu / nhược.** Ưu: chứng cứ có cấu trúc nên kiểm tra được từng claim; trả lời được câu nhiều bước và nhiều đáp án;
confidence của retriever cho biết lúc nào nên từ chối. Nhược: chỉ tốt bằng độ phủ của đồ thị (khoảng 11% câu đáp án không có trong subgraph,
17% không có trong pool ứng viên của GNN); cần xây đồ thị, entity linking và GNN; entity đúng trong đồ thị nhưng sai quan hệ vẫn lọt (strict_QHR).

**3. Hợp với dữ liệu và câu hỏi nào.** Dữ liệu có cấu trúc thực thể quan hệ (Freebase, Wikidata, KG doanh nghiệp, y sinh), câu hỏi factoid
một hoặc nhiều bước, cần giải thích và kiểm chứng câu trả lời.

**4. Demo.** Đi từ GNN-RAG như paper công bố tới CGAR-G: grounded_correct tăng, QHR gần về 0, llm_hit giữ nguyên hoặc tăng.
Cuối phần là ví dụ KG-RAG vẫn sai, cho thấy giới hạn của phương pháp.

In [9]:
specs3 = []
for rd in READERS:
    specs3 += [('GNN-RAG gốc (all, chat)', 'chat', rd, 'gnnrag_all'),
               ('CGAR 5 đường (chat)', 'chat', rd, 'cgar'),
               ('GNN-RAG all + verify', 'chat+verify', rd, 'gnnrag_all'),
               ('CGAR-G (cand + verify)', 'cand+verify', rd, 'cgar_esc'),
               ('GNN-RAG all, LoRA + verify', 'lora-cand+verify', rd, 'gnnrag_all'),
               ('CGAR-G, LoRA + verify', 'lora-cand+verify', rd, 'cgar_esc')]
T3 = table(specs3, cols=['llm_hit', 'llm_f1', 'grounded_correct', 'QHR', 'strict_QHR', 'AURC_incorrect', '#paths', 'tokens'])

llm_hit  llm_f1  grounded_correct    QHR  strict_QHR  AURC_incorrect  #paths   tokens
reader       cấu hình                                                                                                         
Qwen2.5-7B   GNN-RAG gốc (all, chat)       0.765   0.625             0.722  0.114       0.241           0.157   8.476  239.068
             CGAR 5 đường (chat)           0.757   0.583             0.714  0.120       0.236           0.105   3.462  123.070
             GNN-RAG all + verify          0.770   0.627             0.769  0.002       0.195           0.165   8.476  239.068
             CGAR-G (cand + verify)        0.791   0.630             0.790  0.002       0.245           0.119   7.422  265.872
             GNN-RAG all, LoRA + verify    0.800   0.677             0.799  0.002       0.234           0.207   8.476  299.827
             CGAR-G, LoRA + verify         0.797   0.666             0.796  0.002       0.236           0.120   7.422  265.872
Llama-3.1-8B GNN-RAG gốc (all, chat)       0.805   0.630             0.744  0.140       0.283           0.176   8.476  232.797
             CGAR 5 đường (chat)           0.772   0.588             0.713  0.152       0.275           0.119   3.462  120.697
             GNN-RAG all + verify          0.798   0.639             0.796  0.002       0.240           0.197   8.476  232.797
             CGAR-G (cand + verify)        0.804   0.633             0.804  0.001       0.257           0.101   7.422  259.157
             GNN-RAG all, LoRA + verify    0.810   0.692             0.809  0.002       0.231           0.204   8.476  291.941
             CGAR-G, LoRA + verify         0.800   0.676             0.799  0.002       0.228           0.121   7.422  259.157

**Biết lúc nào nên từ chối.** LLM đã fine-tune gần như không bao giờ tự từ chối, nên cần một tín hiệu bên ngoài. Confidence của CGAR xếp hạng
được câu dễ sai: nếu chỉ trả lời 50% câu CGAR tự tin nhất, tỷ lệ bịa và tỷ lệ sai thấp hơn GNN-RAG trả lời mọi câu.

In [10]:
rows = []
for rd in READERS:
    h = hsummary('chat', rd)
    if 'cgar' in h and 'gnnrag_all' in h and 'halluc_risk_at' in h['cgar']:
        rows.append({'reader': SHORT.get(rd, rd),
                     'bịa: GNN-RAG trả lời 100%': h['gnnrag_all']['halluc_risk_at']['1.0'],
                     'bịa: CGAR trả lời 50% tự tin nhất': h['cgar']['halluc_risk_at']['0.5'],
                     'sai: GNN-RAG trả lời 100%': h['gnnrag_all']['incorrect_risk_at']['1.0'],
                     'sai: CGAR trả lời 50% tự tin nhất': h['cgar']['incorrect_risk_at']['0.5']})
display(pd.DataFrame(rows).set_index('reader').astype(float).round(3)) if rows else print('(chưa có kết quả P9)')

,bịa: GNN-RAG trả lời 100%,bịa: CGAR trả lời 50% tự tin nhất,sai: GNN-RAG trả lời 100%,sai: CGAR trả lời 50% tự tin nhất
reader,,,,
Qwen2.5-7B,0.114,0.080,0.164,0.122
Llama-3.1-8B,0.140,0.094,0.171,0.135


In [11]:
rd = READERS[0]
ac, lc, lr = claims('chat', rd, 'gnnrag_all'), claims('lora-cand+verify', rd, 'cgar_esc'), records('lora-cand+verify', rd, 'cgar_esc')
SPECS_SHOW = [('GNN-RAG gốc (chat)', 'chat', rd, 'gnnrag_all'), ('CGAR-G, LoRA + verify', 'lora-cand+verify', rd, 'cgar_esc')]
EX3 = pick(lambda q: ac.get(q, {}).get('hallucinated') == 1 and lc.get(q, {}).get('grounded_correct') == 1, k=2)
print('Ví dụ GNN-RAG gốc bịa, CGAR-G trả lời đúng và có chứng cứ\n')
for qid in EX3: show_question(qid, SPECS_SHOW); print()
EX3_FAIL = (pick(lambda q: lc.get(q, {}).get('error_category') == 'missing_kg_evidence', k=1)
            + pick(lambda q: lc.get(q, {}).get('n_uncertain', 0) > 0 and lr.get(q, {}).get('llm_hit') == 0, k=1))
print('Ví dụ KG-RAG vẫn sai: đáp án không có trong pool ứng viên của GNN, hoặc chọn entity có thật nhưng sai quan hệ\n')
for qid in EX3_FAIL: show_question(qid, SPECS_SHOW); print()

Ví dụ GNN-RAG gốc bịa, CGAR-G trả lời đúng và có chứng cứ

[WebQTest-1009] where did george herbert walker bush go to college
   gold: ['University Yale']
-- GNN-RAG gốc (chat) | Qwen2.5-7B | hit=0 | đáp án có trong context=1
   trả lời: - Yale University - Phillips Academy
     claim 'Yale University'                                UNSUPPORTED absent               gold=False
     claim 'Phillips Academy'                               SUPPORTED   path_endpoint        gold=False
   loại lỗi: reader_unsupported_generation
-- CGAR-G, LoRA + verify | Qwen2.5-7B | hit=1 | đáp án có trong context=1
   trả lời: University Yale Phillips Academy
     claim 'University Yale'                                SUPPORTED   path_endpoint        gold=True
     claim 'Phillips Academy'                               SUPPORTED   path_endpoint        gold=False
   loại lỗi: ok_grounded

[WebQTest-1021] what happened at benghazi
   gold: ['2013 Benghazi conflict', 'First Battle of Benghazi', 'Operation Digni

## Kết luận · So sánh ba loại

Bảng số: một đại diện cho mỗi loại, cùng 1000 câu, cùng reader. Ghi ra `results/headline_tables.csv` để đưa vào slide.

In [12]:
specs4 = []
for rd in READERS:
    specs4 += [('1. LLM thuần', 'chat', rd, 'no_retrieval'),
               ('2. RAG dense top-5', 'chat', rd, 'sbert_top5'),
               ('3a. KG-RAG: GNN-RAG gốc', 'chat', rd, 'gnnrag_all'),
               ('3b. KG-RAG: CGAR-G + LoRA + verify', 'lora-cand+verify', rd, 'cgar_esc')]
T4 = table(specs4, cols=['llm_hit', 'grounded_correct', 'QHR', 'strict_QHR', '#paths', 'tokens'])
parts = {k: v for k, v in (('1_llm', T1), ('2_rag', T2), ('3_kgrag', T3), ('4_ket_luan', T4)) if len(v)}
if parts:
    os.makedirs(OUT_DIR, exist_ok=True)
    pd.concat(parts, names=['phần']).round(4).to_csv(f'{OUT_DIR}/headline_tables.csv')
    print('saved', f'{OUT_DIR}/headline_tables.csv')

llm_hit  grounded_correct    QHR  strict_QHR  #paths   tokens
reader       cấu hình                                                                                         
Qwen2.5-7B   1. LLM thuần                          0.474               NaN    NaN         NaN   0.000   41.342
             2. RAG dense top-5                    0.755             0.709  0.118       0.254   3.462  124.498
             3a. KG-RAG: GNN-RAG gốc               0.765             0.722  0.114       0.241   8.476  239.068
             3b. KG-RAG: CGAR-G + LoRA + verify    0.797             0.796  0.002       0.236   7.422  265.872
Llama-3.1-8B 1. LLM thuần                          0.588               NaN    NaN         NaN   0.000   41.259
             2. RAG dense top-5                    0.768             0.704  0.150       0.289   3.462  121.593
             3a. KG-RAG: GNN-RAG gốc               0.805             0.744  0.140       0.283   8.476  232.797
             3b. KG-RAG: CGAR-G + LoRA + verify    0.800             0.799  0.002       0.228   7.422  259.157

saved /content/drive/MyDrive/ThS/Research/colab/seminar2_kgrag/results/headline_tables.csv


| | LLM thuần | RAG (dense top-K) | KG-RAG (GNN-RAG, CGAR-G) |
|---|---|---|---|
| Accuracy trên hỏi đáp tri thức | thấp, phụ thuộc trí nhớ | khá, mất đáp án khi top-K cắt sai | cao nhất, bị chặn bởi độ phủ của đồ thị |
| Hallucination | cao, không kiểm chứng được | giảm, vẫn chèn kiến thức ngoài context | thấp nhất khi có verify; còn lỗi sai quan hệ |
| Chi phí | rẻ nhất | thêm truy hồi và token context | thêm đồ thị, GNN, nhiều token hơn; CGAR giảm số đường |
| Hợp với dữ liệu | nhãn cố định, tín hiệu nằm trong đầu vào (ví dụ mã nguồn PyPI) | văn bản phi cấu trúc, thay đổi thường xuyên | dữ liệu thực thể quan hệ có sẵn đồ thị |
| Hợp với câu hỏi | phân loại, sinh văn bản theo mẫu | một bước, đáp án nằm trong một đoạn | factoid nhiều bước, nhiều đáp án, cần giải thích |

**Thông điệp chính.** Paper PyPI thấy fine-tune hơn RAG vì bài toán là phân loại với tín hiệu nằm ngay trong đầu vào. Trên hỏi đáp tri thức thì
ngược lại: fine-tune không có context chỉ đạt khoảng 0.41 đến 0.45 llm_hit, còn có đường đi trong đồ thị đạt khoảng 0.80. Và cách giảm bịa hiệu quả
nhất không phải bảo LLM "đừng bịa" (prompt grounded làm nó bỏ cuộc) mà là ràng buộc câu trả lời vào chứng cứ kiểm chứng được (verify theo đường đi,
reader học chọn trong danh sách ứng viên), đồng thời dùng confidence của retriever để biết lúc nào nên từ chối.

**Giới hạn cần nói rõ.** Nhãn hallucination là nhãn tự động từ đồ thị, chưa có nhãn người. Verify giảm QHR gần về 0 một phần do cùng luật với
bộ đo, nên phải đọc kèm grounded_correct và strict_QHR. Một seed reader, 1000 câu WebQSP.

## Tra một câu bất kỳ khi trình bày
Đổi `QID` thành mã câu cần xem (ví dụ một mã in ở các ví dụ trên).

In [13]:
QID = (EX3 or EX1 or EX2 or [None])[0]
ALL_SPECS = [('LLM thuần', 'chat', READERS[0], 'no_retrieval'), ('RAG dense top-5', 'chat', READERS[0], 'sbert_top5')] + SPECS_SHOW
if QID: show_question(QID, ALL_SPECS)

[WebQTest-1009] where did george herbert walker bush go to college
   gold: ['University Yale']
-- LLM thuần | Qwen2.5-7B | hit=0 | đáp án có trong context=0
   trả lời: - Yale University - Dartmouth College
-- RAG dense top-5 | Qwen2.5-7B | hit=0 | đáp án có trong context=1
   trả lời: - Yale University - Phillips Academy
     claim 'Yale University'                                UNSUPPORTED absent               gold=False
     claim 'Phillips Academy'                               SUPPORTED   path_endpoint        gold=False
   loại lỗi: reader_unsupported_generation
-- GNN-RAG gốc (chat) | Qwen2.5-7B | hit=0 | đáp án có trong context=1
   trả lời: - Yale University - Phillips Academy
     claim 'Yale University'                                UNSUPPORTED absent               gold=False
     claim 'Phillips Academy'                               SUPPORTED   path_endpoint        gold=False
   loại lỗi: reader_unsupported_generation
-- CGAR-G, LoRA + verify | Qwen2.5-7B | hit=1 | đáp á